# MOB-suite Results: Chromosome/Plasmid + Mobility, per Organism

Standalone notebook — MOB-suite's own two output files only. No RGI/CARD data, no ARG gene names, nothing from `ARG_Chromosome_Plasmid_Mapping.ipynb`.

**What each MOB-suite file actually is:**

- **`contig_report.txt` / `master_contig_report.tsv`** — one row per **contig**. Columns used here: `sample_id` (genome accession), `contig_id` (the literal NCBI nucleotide accession, e.g. `NZ_CP151710.1` — this is what can be pasted into NCBI to independently verify a chromosome/plasmid call), `molecule_type` (`chromosome` or `plasmid`), `primary_cluster_id` (which plasmid this contig belongs to; only populated on plasmid rows).
- **`mobtyper_results.txt` / `master_mobtyper_results.tsv`** — one row per **assembled plasmid**, not per contig (a single plasmid can be built from more than one contig, so this table is coarser than the contig table). Columns used here: `pipeline_sample_id` (genome accession — the correct join key), `primary_cluster_id`, `predicted_mobility` (`conjugative` / `mobilizable` / `non-mobilizable`). **Not** this file's own `sample_id` column — that one is a composite `"<accession>:<primary_cluster_id>"` string (verified against the raw file), which silently breaks this join if used directly.

**The join:** `contig_report` says *which* contigs are plasmid and *which plasmid* each one belongs to; `mobtyper` says, for that plasmid, whether it's mobile. Joining the two on `(sample_id, primary_cluster_id)` gives, per contig, both facts at once.

**Structure:** each of the six ESKAPE organisms is loaded and joined separately, in its own section with its own named variables — no loop, so every intermediate result stays inspectable. Output for each organism is a per-contig table with `contig_id` (a real NCBI accession, for manual cross-checking), `molecule_type`, and `predicted_mobility`.

**Note:** this notebook only defines the pipeline; no cells have been run yet.

## 1. Configuration

In [1]:
# Standard library / third-party imports.
import sys
import os
import pandas as pd
import warnings
import matplotlib
warnings.filterwarnings('ignore')
print(f'Python      : {sys.version.split()[0]}')
print(f'pandas      : {pd.__version__}')
print(f'matplotlib  : {matplotlib.__version__}')
# Project root and the output folder (shared with the ARG-mapping notebook's outputs).
BASE_DIR = '/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family'
RESULTS_DIR = os.path.join(BASE_DIR, 'Results/Mobsuite/Plasmid_classification')
os.makedirs(RESULTS_DIR, exist_ok=True)

Python      : 3.12.12
pandas      : 2.3.3
matplotlib  : 3.10.8


Organism code -> (MOB-suite results directory, full species name).

Only the MOB-suite directory is needed here — this notebook never reads CARD_Results/RGI at all. Folder-name casing is inconsistent across organisms (`kp_Mob_Results` lowercase, `PA_Mob_Results` uppercase, `Ent_Mob_Results` title-case), which is why this is hardcoded rather than derived from the organism code.

In [2]:
# Organism code -> (MOB-suite results dir, full species name).
ORGANISMS = {
    'AB':  ('Results/AB_Mob_Results',  'Acinetobacter baumannii'),
    'EF':  ('Results/EF_Mob_Results',  'Enterococcus faecium'),
    'ENT': ('Results/Ent_Mob_Results', 'Enterobacter spp.'),
    'Kp':  ('Results/kp_Mob_Results',  'Klebsiella pneumoniae'),
    'Pa':  ('Results/PA_Mob_Results',  'Pseudomonas aeruginosa'),
    'SA':  ('Results/SA_Mob_Results',  'Staphylococcus aureus'),
}

## 2. Shared file-loading helpers

Pure file I/O, not analysis — kept as small reusable functions rather than repeated six times. The actual load/join for each organism is done explicitly in its own section below, not inside a loop.

In [3]:
def load_mob_contig_report(mob_dir):
    """
    Load one organism's MOB-suite contig-level chromosome/plasmid calls.

    Reads the pre-aggregated summary/master_contig_report.tsv (one row per contig,
    across every genome for this organism).
    """
    contig_df = pd.read_csv(os.path.join(mob_dir, 'summary', 'master_contig_report.tsv'), sep='\t')
    return contig_df[['sample_id', 'contig_id', 'molecule_type', 'primary_cluster_id']]

In [5]:
def load_mob_mobtyper_results(mob_dir):
    """
    Load one organism's MOB-suite plasmid mobility calls.

    Uses 'pipeline_sample_id' (the plain genome accession), not this file's own 'sample_id'
    column, which is a composite '<accession>:<primary_cluster_id>' string (verified against
    the raw file) that cannot match the plain accession used in the contig table.
    """
    mobtyper_df = pd.read_csv(os.path.join(mob_dir, 'summary', 'master_mobtyper_results.tsv'), sep='\t')
    
    mobtyper_df = mobtyper_df.rename(columns={'pipeline_sample_id': 'sample_id'})
    return mobtyper_df[['sample_id', 'primary_cluster_id', 'predicted_mobility']]

In [4]:
# Fix: load_mob_mobtyper_results() above leaves a duplicate 'sample_id' column.
# The raw file already has its own native 'sample_id' (a composite '<accession>:<primary_cluster_id>'
# string) *and* a separate 'pipeline_sample_id' (the plain accession). Renaming pipeline_sample_id
# to sample_id doesn't remove the original — it just creates two columns with the same label, which
# then breaks the merge in the join cell with "ValueError: The column label 'sample_id' is not unique."
#
# Fix: select only the needed columns FIRST (dropping the native sample_id before it can collide),
# THEN rename pipeline_sample_id -> sample_id. Redefining the function here overrides the version
# above for the rest of this notebook, without editing that cell directly.
def load_mob_mobtyper_results(mob_dir):
    mobtyper_df = pd.read_csv(os.path.join(mob_dir, 'summary', 'master_mobtyper_results.tsv'), sep='\t')
    mobtyper_df = mobtyper_df[['pipeline_sample_id', 'primary_cluster_id', 'predicted_mobility']]
    mobtyper_df = mobtyper_df.rename(columns={'pipeline_sample_id': 'sample_id'})
    return mobtyper_df

## 3. Per-organism: load, join, verify, save

The same steps are repeated explicitly for each of the six organisms below, each in its own section with its own named variables, rather than in a loop.

### 3.1 Acinetobacter baumannii (AB)

In [18]:
import pandas as pd

# Corrected syntax: removed the ')' after the .tsv string
mobcontig_df1 = pd.read_csv("/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/AB_Mob_Results/summary/master_contig_report.tsv", sep='\t')

# Optional: View the first few rows to confirm it loaded correctly
mobcontig_df1.head().style

,pipeline_sample_id,sample_id,molecule_type,primary_cluster_id,secondary_cluster_id,contig_id,size,gc,md5,circularity_status,rep_type(s),rep_type_accession(s),relaxase_type(s),relaxase_type_accession(s),mpf_type,mpf_type_accession(s),orit_type(s),orit_accession(s),predicted_mobility,mash_nearest_neighbor,mash_neighbor_distance,mash_neighbor_identification,repetitive_dna_id,repetitive_dna_type,filtering_reason
0,GCA_014880215.2,GCA_014880215.2,chromosome,-,-,CP081137.1,4010463,0.389097,b5db0eef96fffcbe9d02cc1eef49d206,incomplete,-,-,MOBQ,NZ_ALIR01000019_00003,-,-,-,-,-,-,-,-,-,-,-
1,GCA_014880215.2,GCA_014880215.2,plasmid,AC163,AL241,CP081138.1,11323,0.345138,6c9386d42e09431296bb73da458c6e69,incomplete,rep_cluster_1172,000454__KY202458_00008,-,-,-,-,-,-,-,NC_021489,0.00433199,Acinetobacter baumannii,-,-,-
2,GCA_014881055.2,GCA_014881055.2,chromosome,-,-,CP081139.1,3869345,0.391293,dc6b523de75ca3a9e838858c6cae83bb,incomplete,rep_cluster_463,001812__CP017939,-,-,-,-,-,-,-,-,-,-,-,-,-
3,GCA_014881055.2,GCA_014881055.2,plasmid,AB082,AJ649,CP081140.1,73194,0.336790,a4626d1825fcbf6cb8a7bf494675a318,incomplete,rep_cluster_734,002211__NC_010606_00001,MOBF,NC_017163_00021,"MPF_F,MPF_F,MPF_F,MPF_F,MPF_F,MPF_F,MPF_F,MPF_F,MPF_F,MPF_T,MPF_T","NC_010606_00071,NC_010606_00072,NC_010606_00075,NC_010606_00078,NC_010606_00079,NC_010606_00080,NZ_AEOY01000095_00077,NC_017163_00052,NC_017163_00053,NC_017163_00020,NC_010606_00076",-,-,-,CP026706,9.55247e-05,Acinetobacter baumannii,-,-,-
4,GCA_014881055.2,GCA_014881055.2,plasmid,AG958,AR014,CP081141.1,68153,0.362860,75a13fa12b1511c7741a9af2164f91ea,incomplete,rep_cluster_386,001711__CP013925_00062,MOBF,CP013925_00029,-,-,-,-,-,CP013925,0.0183741,Acinetobacter baumannii,-,-,-


In [19]:
import pandas as pd

# Corrected syntax: removed the ')' after the .tsv string
mobtyper_df1 = pd.read_csv("/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/Results/AB_Mob_Results/summary/master_mobtyper_results.tsv", sep='\t')

# Optional: View the first few rows to confirm it loaded correctly
mobtyper_df1.head().style

,pipeline_sample_id,sample_id,num_contigs,size,gc,md5,rep_type(s),rep_type_accession(s),relaxase_type(s),relaxase_type_accession(s),mpf_type,mpf_type_accession(s),orit_type(s),orit_accession(s),predicted_mobility,mash_nearest_neighbor,mash_neighbor_distance,mash_neighbor_identification,primary_cluster_id,secondary_cluster_id,predicted_host_range_overall_rank,predicted_host_range_overall_name,observed_host_range_ncbi_rank,observed_host_range_ncbi_name,reported_host_range_lit_rank,reported_host_range_lit_name,associated_pmid(s)
0,GCA_014880215.2,GCA_014880215.2:AC163,1,11323,0.345138,6c9386d42e09431296bb73da458c6e69,rep_cluster_1172,000454__KY202458_00008,-,-,-,-,-,-,non-mobilizable,NC_021489,0.004332,Acinetobacter baumannii,AC163,AL241,genus,Acinetobacter,genus,Acinetobacter,-,-,-
1,GCA_014881055.2,GCA_014881055.2:AB082,1,73194,0.336790,a4626d1825fcbf6cb8a7bf494675a318,rep_cluster_734,002211__NC_010606_00001,MOBF,NC_017163_00021,MPF_F,"NC_010606_00071,NC_010606_00072,NC_010606_00075,NC_010606_00078,NC_010606_00079,NC_010606_00080,NZ_AEOY01000095_00077,NC_017163_00052,NC_017163_00053,NC_017163_00020,NC_010606_00076",-,-,conjugative,CP026706,0.000096,Acinetobacter baumannii,AB082,AJ649,genus,Acinetobacter,genus,Acinetobacter,-,-,-
2,GCA_014881055.2,GCA_014881055.2:AG958,1,68153,0.362860,75a13fa12b1511c7741a9af2164f91ea,rep_cluster_386,001711__CP013925_00062,MOBF,CP013925_00029,-,-,-,-,mobilizable,CP013925,0.018374,Acinetobacter baumannii,AG958,AR014,genus,Acinetobacter,genus,Acinetobacter,-,-,-
3,GCA_014881055.2,GCA_014881055.2:AC160,1,10967,0.369199,a3791e9dcc426837e8ed67186dccd5ab,rep_cluster_1172,000453__KU549174_00001,MOBQ,LN865144_00010,-,-,-,-,mobilizable,CP035053,0.000000,Acinetobacter baumannii,AC160,AL236,genus,Acinetobacter,genus,Acinetobacter,-,-,-
4,GCA_014882805.2,GCA_014882805.2:AB113,1,106967,0.426449,cad630f568ab0527ccddff89cb7d8f90,-,-,-,-,-,-,-,-,non-mobilizable,CP027608,0.000000,Acinetobacter baumannii,AB113,AJ688,genus,Acinetobacter,genus,Acinetobacter,-,-,-


In [5]:
# Load Acinetobacter baumannii (AB) MOB-suite tables into their own named variables.
AB_contig_df = load_mob_contig_report(os.path.join(BASE_DIR, 'Results/AB_Mob_Results'))
AB_mobtyper_df = load_mob_mobtyper_results(os.path.join(BASE_DIR, 'Results/AB_Mob_Results'))

print(f'AB: {len(AB_contig_df)} contigs, {len(AB_mobtyper_df)} assembled plasmids loaded')

AB: 2714 contigs, 1234 assembled plasmids loaded


In [6]:
# Checkpoint: molecule_type breakdown, before joining anything.
AB_contig_df['molecule_type'].value_counts()

molecule_type
plasmid       1401
chromosome    1313
Name: count, dtype: int64

In [7]:
# Checkpoint: predicted_mobility breakdown, before joining anything.
AB_mobtyper_df['predicted_mobility'].value_counts()

predicted_mobility
conjugative        430
non-mobilizable    416
mobilizable        388
Name: count, dtype: int64

In [8]:
# Step 1: Extract the column names from your dataframe and convert to a list
column_headers = AB_contig_df.columns.tolist()

# Step 2: Print a quick summary of how many columns you have
print(f"Total columns: {len(column_headers)}\n")

# Step 3: Loop through and print each column name on a new line
for col in column_headers:
    print(col)

Total columns: 4

sample_id
contig_id
molecule_type
primary_cluster_id


In [9]:
# Step 1: Extract the column names from your dataframe and convert to a list
column_headers = AB_mobtyper_df.columns.tolist()

# Step 2: Print a quick summary of how many columns you have
print(f"Total columns: {len(column_headers)}\n")

# Step 3: Loop through and print each column name on a new line
for col in column_headers:
    print(col)

Total columns: 3

sample_id
primary_cluster_id
predicted_mobility


In [10]:
# Join contig-level chromosome/plasmid calls to plasmid-level mobility calls,
# on (sample_id, primary_cluster_id).
AB_contig_plasmid_mobility_df = AB_contig_df.merge(
    AB_mobtyper_df,
    on=['sample_id', 'primary_cluster_id'],
    how='left',
)
# Chromosomal contigs have no primary_cluster_id and so no mobility class; label them explicitly.
AB_contig_plasmid_mobility_df.loc[
    AB_contig_plasmid_mobility_df['molecule_type'] != 'plasmid', 'predicted_mobility'
] = 'n/a (chromosomal)'

print(f'AB: {len(AB_contig_plasmid_mobility_df)} contigs in AB_contig_plasmid_mobility_df')

AB: 2714 contigs in AB_contig_plasmid_mobility_df


In [11]:
# Verify the join: how many plasmid contigs still have a blank predicted_mobility? Expect 0.
AB_plasmid_rows = AB_contig_plasmid_mobility_df[AB_contig_plasmid_mobility_df['molecule_type'] == 'plasmid']
AB_n_blank_mobility = int(AB_plasmid_rows['predicted_mobility'].isna().sum())
print(f'AB: {len(AB_plasmid_rows)} plasmid contigs, {AB_n_blank_mobility} with blank predicted_mobility')

# Sample rows with their contig_id — real NCBI accessions, ready for manual cross-checking.
AB_plasmid_rows[['sample_id', 'contig_id', 'molecule_type', 'primary_cluster_id', 'predicted_mobility']].head(10)

AB: 1401 plasmid contigs, 0 with blank predicted_mobility


,sample_id,contig_id,molecule_type,primary_cluster_id,predicted_mobility
1,GCA_014880215.2,CP081138.1,plasmid,AC163,non-mobilizable
3,GCA_014881055.2,CP081140.1,plasmid,AB082,conjugative
4,GCA_014881055.2,CP081141.1,plasmid,AG958,mobilizable
5,GCA_014881055.2,CP081142.1,plasmid,AC160,mobilizable
9,GCA_014882805.2,CP081146.1,plasmid,AB113,non-mobilizable
10,GCA_014882805.2,CP081147.1,plasmid,AC160,mobilizable
11,GCA_021387855.2,CP097877.1,plasmid,AB114,mobilizable
13,GCA_021387855.2,CP097879.1,plasmid,AC163,non-mobilizable
14,GCA_021387855.2,CP097880.1,plasmid,AB114,mobilizable
16,GCA_021387885.2,CP097876.1,plasmid,AB114,mobilizable


In [12]:
# Save this organism's per-contig chromosome/plasmid + mobility table.
AB_contig_plasmid_mobility_df.to_csv(
    os.path.join(RESULTS_DIR, 'AB_Contig_Chromosome_Plasmid_Mobility.csv'), index=False
)

### 3.2 Enterococcus faecium (EF)

In [13]:
# Load Enterococcus faecium (EF) MOB-suite tables into their own named variables.
EF_contig_df = load_mob_contig_report(os.path.join(BASE_DIR, 'Results/EF_Mob_Results'))
EF_mobtyper_df = load_mob_mobtyper_results(os.path.join(BASE_DIR, 'Results/EF_Mob_Results'))

print(f'EF: {len(EF_contig_df)} contigs, {len(EF_mobtyper_df)} assembled plasmids loaded')

EF: 4425 contigs, 2416 assembled plasmids loaded


In [14]:
# Checkpoint: molecule_type breakdown, before joining anything.
EF_contig_df['molecule_type'].value_counts()

molecule_type
plasmid       3208
chromosome    1217
Name: count, dtype: int64

In [15]:
# Checkpoint: predicted_mobility breakdown, before joining anything.
EF_mobtyper_df['predicted_mobility'].value_counts()

predicted_mobility
non-mobilizable    1045
mobilizable         849
conjugative         522
Name: count, dtype: int64

In [16]:
# Join contig-level chromosome/plasmid calls to plasmid-level mobility calls,
# on (sample_id, primary_cluster_id).
EF_contig_plasmid_mobility_df = EF_contig_df.merge(
    EF_mobtyper_df,
    on=['sample_id', 'primary_cluster_id'],
    how='left',
)
# Chromosomal contigs have no primary_cluster_id and so no mobility class; label them explicitly.
EF_contig_plasmid_mobility_df.loc[
    EF_contig_plasmid_mobility_df['molecule_type'] != 'plasmid', 'predicted_mobility'
] = 'n/a (chromosomal)'

print(f'EF: {len(EF_contig_plasmid_mobility_df)} contigs in EF_contig_plasmid_mobility_df')

EF: 4425 contigs in EF_contig_plasmid_mobility_df


In [17]:
# Verify the join: how many plasmid contigs still have a blank predicted_mobility? Expect 0.
EF_plasmid_rows = EF_contig_plasmid_mobility_df[EF_contig_plasmid_mobility_df['molecule_type'] == 'plasmid']
EF_n_blank_mobility = int(EF_plasmid_rows['predicted_mobility'].isna().sum())
print(f'EF: {len(EF_plasmid_rows)} plasmid contigs, {EF_n_blank_mobility} with blank predicted_mobility')

# Sample rows with their contig_id — real NCBI accessions, ready for manual cross-checking.
EF_plasmid_rows[['sample_id', 'contig_id', 'molecule_type', 'primary_cluster_id', 'predicted_mobility']].head(10)

EF: 3208 plasmid contigs, 0 with blank predicted_mobility


,sample_id,contig_id,molecule_type,primary_cluster_id,predicted_mobility
1,GCA_015770055.2,JADUTS020000002.1,plasmid,AC731,conjugative
2,GCA_015770055.2,JADUTS020000003.1,plasmid,AC731,conjugative
3,GCA_015770055.2,JADUTS020000004.1,plasmid,AB173,mobilizable
4,GCA_015770055.2,JADUTS020000005.1,plasmid,AA894,mobilizable
5,GCA_015770055.2,JADUTS020000006.1,plasmid,AB173,mobilizable
6,GCA_015770055.2,JADUTS020000007.1,plasmid,AB173,mobilizable
8,GCA_015993525.2,JADUEF020000002.1,plasmid,AB369,non-mobilizable
9,GCA_015993525.2,JADUEF020000003.1,plasmid,AD910,conjugative
10,GCA_015993525.2,JADUEF020000004.1,plasmid,AB173,mobilizable
12,GCA_015994025.2,JADUFD020000002.1,plasmid,AC731,conjugative


In [18]:
# Save this organism's per-contig chromosome/plasmid + mobility table.
EF_contig_plasmid_mobility_df.to_csv(
    os.path.join(RESULTS_DIR, 'EF_Contig_Chromosome_Plasmid_Mobility.csv'), index=False
)

### 3.3 Enterobacter spp. (ENT)

In [19]:
# Load Enterobacter spp. (ENT) MOB-suite tables into their own named variables.
ENT_contig_df = load_mob_contig_report(os.path.join(BASE_DIR, 'Results/Ent_Mob_Results'))
ENT_mobtyper_df = load_mob_mobtyper_results(os.path.join(BASE_DIR, 'Results/Ent_Mob_Results'))

print(f'ENT: {len(ENT_contig_df)} contigs, {len(ENT_mobtyper_df)} assembled plasmids loaded')

ENT: 2316 contigs, 1336 assembled plasmids loaded


In [20]:
# Checkpoint: molecule_type breakdown, before joining anything.
ENT_contig_df['molecule_type'].value_counts()

molecule_type
plasmid       1393
chromosome     923
Name: count, dtype: int64

In [21]:
# Checkpoint: predicted_mobility breakdown, before joining anything.
ENT_mobtyper_df['predicted_mobility'].value_counts()

predicted_mobility
conjugative        598
mobilizable        493
non-mobilizable    245
Name: count, dtype: int64

In [22]:
# Join contig-level chromosome/plasmid calls to plasmid-level mobility calls,
# on (sample_id, primary_cluster_id).
ENT_contig_plasmid_mobility_df = ENT_contig_df.merge(
    ENT_mobtyper_df,
    on=['sample_id', 'primary_cluster_id'],
    how='left',
)
# Chromosomal contigs have no primary_cluster_id and so no mobility class; label them explicitly.
ENT_contig_plasmid_mobility_df.loc[
    ENT_contig_plasmid_mobility_df['molecule_type'] != 'plasmid', 'predicted_mobility'
] = 'n/a (chromosomal)'

print(f'ENT: {len(ENT_contig_plasmid_mobility_df)} contigs in ENT_contig_plasmid_mobility_df')

ENT: 2316 contigs in ENT_contig_plasmid_mobility_df


In [23]:
# Verify the join: how many plasmid contigs still have a blank predicted_mobility? Expect 0.
ENT_plasmid_rows = ENT_contig_plasmid_mobility_df[ENT_contig_plasmid_mobility_df['molecule_type'] == 'plasmid']
ENT_n_blank_mobility = int(ENT_plasmid_rows['predicted_mobility'].isna().sum())
print(f'ENT: {len(ENT_plasmid_rows)} plasmid contigs, {ENT_n_blank_mobility} with blank predicted_mobility')

# Sample rows with their contig_id — real NCBI accessions, ready for manual cross-checking.
ENT_plasmid_rows[['sample_id', 'contig_id', 'molecule_type', 'primary_cluster_id', 'predicted_mobility']].head(10)

ENT: 1393 plasmid contigs, 0 with blank predicted_mobility


,sample_id,contig_id,molecule_type,primary_cluster_id,predicted_mobility
1,GCA_018597075.2,CP128418.1,plasmid,AB169,conjugative
2,GCA_018597075.2,CP128419.1,plasmid,AA739,conjugative
3,GCA_018597075.2,CP128420.1,plasmid,AB040,non-mobilizable
5,GCA_027577865.1,CP114145.1,plasmid,AA425,conjugative
6,GCA_027577865.1,CP114146.1,plasmid,AA046,conjugative
7,GCA_027577865.1,CP114147.1,plasmid,AA002,conjugative
9,GCA_030179315.1,CP109735.1,plasmid,AA824,non-mobilizable
10,GCA_030179315.1,CP109736.1,plasmid,AB978,mobilizable
11,GCA_030179315.1,CP109737.1,plasmid,AA739,conjugative
14,GCA_030718955.1,CP132210.1,plasmid,AA621,conjugative


In [24]:
# Save this organism's per-contig chromosome/plasmid + mobility table.
ENT_contig_plasmid_mobility_df.to_csv(
    os.path.join(RESULTS_DIR, 'ENT_Contig_Chromosome_Plasmid_Mobility.csv'), index=False
)

### 3.4 Klebsiella pneumoniae (Kp)

In [25]:
# Load Klebsiella pneumoniae (Kp) MOB-suite tables into their own named variables.
Kp_contig_df = load_mob_contig_report(os.path.join(BASE_DIR, 'Results/kp_Mob_Results'))
Kp_mobtyper_df = load_mob_mobtyper_results(os.path.join(BASE_DIR, 'Results/kp_Mob_Results'))

print(f'Kp: {len(Kp_contig_df)} contigs, {len(Kp_mobtyper_df)} assembled plasmids loaded')

Kp: 17423 contigs, 10875 assembled plasmids loaded


In [26]:
# Checkpoint: molecule_type breakdown, before joining anything.
Kp_contig_df['molecule_type'].value_counts()

molecule_type
plasmid       11899
chromosome     5524
Name: count, dtype: int64

In [27]:
# Checkpoint: predicted_mobility breakdown, before joining anything.
Kp_mobtyper_df['predicted_mobility'].value_counts()

predicted_mobility
conjugative        4589
non-mobilizable    3337
mobilizable        2949
Name: count, dtype: int64

In [28]:
# Join contig-level chromosome/plasmid calls to plasmid-level mobility calls,
# on (sample_id, primary_cluster_id).
Kp_contig_plasmid_mobility_df = Kp_contig_df.merge(
    Kp_mobtyper_df,
    on=['sample_id', 'primary_cluster_id'],
    how='left',
)
# Chromosomal contigs have no primary_cluster_id and so no mobility class; label them explicitly.
Kp_contig_plasmid_mobility_df.loc[
    Kp_contig_plasmid_mobility_df['molecule_type'] != 'plasmid', 'predicted_mobility'
] = 'n/a (chromosomal)'

print(f'Kp: {len(Kp_contig_plasmid_mobility_df)} contigs in Kp_contig_plasmid_mobility_df')

Kp: 17423 contigs in Kp_contig_plasmid_mobility_df


In [29]:
# Verify the join: how many plasmid contigs still have a blank predicted_mobility? Expect 0.
Kp_plasmid_rows = Kp_contig_plasmid_mobility_df[Kp_contig_plasmid_mobility_df['molecule_type'] == 'plasmid']
Kp_n_blank_mobility = int(Kp_plasmid_rows['predicted_mobility'].isna().sum())
print(f'Kp: {len(Kp_plasmid_rows)} plasmid contigs, {Kp_n_blank_mobility} with blank predicted_mobility')

# Sample rows with their contig_id — real NCBI accessions, ready for manual cross-checking.
Kp_plasmid_rows[['sample_id', 'contig_id', 'molecule_type', 'primary_cluster_id', 'predicted_mobility']].head(10)

Kp: 11899 plasmid contigs, 0 with blank predicted_mobility


,sample_id,contig_id,molecule_type,primary_cluster_id,predicted_mobility
1,GCA_012972045.1,CP052609.1,plasmid,AA277,conjugative
3,GCA_012972045.1,CP052611.1,plasmid,AA110,non-mobilizable
12,GCA_012972055.1,CP052597.1,plasmid,AA277,conjugative
14,GCA_012972055.1,CP052599.1,plasmid,AA277,conjugative
15,GCA_012972055.1,CP052600.1,plasmid,AD158,conjugative
16,GCA_012972055.1,CP052601.1,plasmid,AA110,non-mobilizable
24,GCA_012972085.1,CP052620.1,plasmid,AA275,conjugative
25,GCA_012972085.1,CP052621.1,plasmid,AC125,non-mobilizable
27,GCA_012972085.1,CP052623.1,plasmid,AA941,mobilizable
28,GCA_012972085.1,CP052624.1,plasmid,AA117,non-mobilizable


In [30]:
# Save this organism's per-contig chromosome/plasmid + mobility table.
Kp_contig_plasmid_mobility_df.to_csv(
    os.path.join(RESULTS_DIR, 'Kp_Contig_Chromosome_Plasmid_Mobility.csv'), index=False
)

### 3.5 Pseudomonas aeruginosa (Pa)

In [31]:
# Load Pseudomonas aeruginosa (Pa) MOB-suite tables into their own named variables.
Pa_contig_df = load_mob_contig_report(os.path.join(BASE_DIR, 'Results/PA_Mob_Results'))
Pa_mobtyper_df = load_mob_mobtyper_results(os.path.join(BASE_DIR, 'Results/PA_Mob_Results'))

print(f'Pa: {len(Pa_contig_df)} contigs, {len(Pa_mobtyper_df)} assembled plasmids loaded')

Pa: 1889 contigs, 236 assembled plasmids loaded


In [32]:
# Checkpoint: molecule_type breakdown, before joining anything.
Pa_contig_df['molecule_type'].value_counts()

molecule_type
chromosome    1647
plasmid        242
Name: count, dtype: int64

In [33]:
# Checkpoint: predicted_mobility breakdown, before joining anything.
Pa_mobtyper_df['predicted_mobility'].value_counts()

predicted_mobility
non-mobilizable    165
mobilizable         43
conjugative         28
Name: count, dtype: int64

In [34]:
# Join contig-level chromosome/plasmid calls to plasmid-level mobility calls,
# on (sample_id, primary_cluster_id).
Pa_contig_plasmid_mobility_df = Pa_contig_df.merge(
    Pa_mobtyper_df,
    on=['sample_id', 'primary_cluster_id'],
    how='left',
)
# Chromosomal contigs have no primary_cluster_id and so no mobility class; label them explicitly.
Pa_contig_plasmid_mobility_df.loc[
    Pa_contig_plasmid_mobility_df['molecule_type'] != 'plasmid', 'predicted_mobility'
] = 'n/a (chromosomal)'

print(f'Pa: {len(Pa_contig_plasmid_mobility_df)} contigs in Pa_contig_plasmid_mobility_df')

Pa: 1889 contigs in Pa_contig_plasmid_mobility_df


In [35]:
# Verify the join: how many plasmid contigs still have a blank predicted_mobility? Expect 0.
Pa_plasmid_rows = Pa_contig_plasmid_mobility_df[Pa_contig_plasmid_mobility_df['molecule_type'] == 'plasmid']
Pa_n_blank_mobility = int(Pa_plasmid_rows['predicted_mobility'].isna().sum())
print(f'Pa: {len(Pa_plasmid_rows)} plasmid contigs, {Pa_n_blank_mobility} with blank predicted_mobility')

# Sample rows with their contig_id — real NCBI accessions, ready for manual cross-checking.
Pa_plasmid_rows[['sample_id', 'contig_id', 'molecule_type', 'primary_cluster_id', 'predicted_mobility']].head(10)

Pa: 242 plasmid contigs, 0 with blank predicted_mobility


,sample_id,contig_id,molecule_type,primary_cluster_id,predicted_mobility
26,GCA_021664795.2,CP132106.1,plasmid,AF165,mobilizable
45,GCA_023615765.2,CP124677.1,plasmid,AF747,mobilizable
58,GCA_028366535.3,CP137941.1,plasmid,AG035,non-mobilizable
71,GCA_030366015.2,CP138392.1,plasmid,AE788,non-mobilizable
73,GCA_030368395.2,CP138390.1,plasmid,AE788,non-mobilizable
79,GCA_033541995.2,CP138579.1,plasmid,AE788,non-mobilizable
117,GCF_000988505.1,NZ_CP011370.1,plasmid,AG036,conjugative
158,GCF_001870265.1,NZ_CP015000.1,plasmid,AE787,non-mobilizable
162,GCF_002085755.1,NZ_CP020602.1,plasmid,AC741,mobilizable
173,GCF_002215345.1,NZ_CP017294.1,plasmid,AC884,conjugative


In [36]:
# Save this organism's per-contig chromosome/plasmid + mobility table.
Pa_contig_plasmid_mobility_df.to_csv(
    os.path.join(RESULTS_DIR, 'Pa_Contig_Chromosome_Plasmid_Mobility.csv'), index=False
)

### 3.6 Staphylococcus aureus (SA)

In [37]:
# Load Staphylococcus aureus (SA) MOB-suite tables into their own named variables.
SA_contig_df = load_mob_contig_report(os.path.join(BASE_DIR, 'Results/SA_Mob_Results'))
SA_mobtyper_df = load_mob_mobtyper_results(os.path.join(BASE_DIR, 'Results/SA_Mob_Results'))

print(f'SA: {len(SA_contig_df)} contigs, {len(SA_mobtyper_df)} assembled plasmids loaded')

SA: 4246 contigs, 1639 assembled plasmids loaded


In [38]:
# Checkpoint: molecule_type breakdown, before joining anything.
SA_contig_df['molecule_type'].value_counts()

molecule_type
chromosome    2485
plasmid       1761
Name: count, dtype: int64

In [39]:
# Checkpoint: predicted_mobility breakdown, before joining anything.
SA_mobtyper_df['predicted_mobility'].value_counts()

predicted_mobility
mobilizable        1259
non-mobilizable     236
conjugative         144
Name: count, dtype: int64

In [40]:
# Join contig-level chromosome/plasmid calls to plasmid-level mobility calls,
# on (sample_id, primary_cluster_id).
SA_contig_plasmid_mobility_df = SA_contig_df.merge(
    SA_mobtyper_df,
    on=['sample_id', 'primary_cluster_id'],
    how='left',
)
# Chromosomal contigs have no primary_cluster_id and so no mobility class; label them explicitly.
SA_contig_plasmid_mobility_df.loc[
    SA_contig_plasmid_mobility_df['molecule_type'] != 'plasmid', 'predicted_mobility'
] = 'n/a (chromosomal)'

print(f'SA: {len(SA_contig_plasmid_mobility_df)} contigs in SA_contig_plasmid_mobility_df')

SA: 4246 contigs in SA_contig_plasmid_mobility_df


In [41]:
# Verify the join: how many plasmid contigs still have a blank predicted_mobility? Expect 0.
SA_plasmid_rows = SA_contig_plasmid_mobility_df[SA_contig_plasmid_mobility_df['molecule_type'] == 'plasmid']
SA_n_blank_mobility = int(SA_plasmid_rows['predicted_mobility'].isna().sum())
print(f'SA: {len(SA_plasmid_rows)} plasmid contigs, {SA_n_blank_mobility} with blank predicted_mobility')

# Sample rows with their contig_id — real NCBI accessions, ready for manual cross-checking.
SA_plasmid_rows[['sample_id', 'contig_id', 'molecule_type', 'primary_cluster_id', 'predicted_mobility']].head(10)

SA: 1761 plasmid contigs, 0 with blank predicted_mobility


,sample_id,contig_id,molecule_type,primary_cluster_id,predicted_mobility
1,GCA_008934005.1,CP030491.1,plasmid,AA083,conjugative
2,GCA_008934005.1,CP030492.1,plasmid,AA840,mobilizable
4,GCA_008934055.1,CP030513.1,plasmid,AA840,mobilizable
6,GCA_008934075.1,CP030494.1,plasmid,AA840,mobilizable
9,GCA_008934095.1,CP030442.1,plasmid,AA840,mobilizable
11,GCA_008934155.1,CP030497.1,plasmid,AA839,mobilizable
15,GCA_008934195.1,CP030477.1,plasmid,AA840,mobilizable
17,GCA_008934215.1,CP030444.1,plasmid,AA840,mobilizable
19,GCA_008934235.1,CP030466.1,plasmid,AA840,mobilizable
22,GCA_008934275.1,CP030468.1,plasmid,AA083,conjugative


In [42]:
# Save this organism's per-contig chromosome/plasmid + mobility table.
SA_contig_plasmid_mobility_df.to_csv(
    os.path.join(RESULTS_DIR, 'SA_Contig_Chromosome_Plasmid_Mobility.csv'), index=False
)